# Chapter 2 — Exercise Solutions## Probability Prerequisites[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)> Attempt exercises first in `02_exercises.ipynb`.

### Solution 2.1: Expected Value of Slot Machine

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

probs   = [0.1,  0.3,  0.6]
payouts = [10,   2,    -1]

# a) Analytical expectation: E[X] = sum(p * x)
E_analytical = sum(p * x for p, x in zip(probs, payouts))
print(f"a) Analytical E[X] = {E_analytical:.4f}")
# = 0.1*10 + 0.3*2 + 0.6*(-1) = 1.0 + 0.6 - 0.6 = 1.0

# b) Simulation
np.random.seed(42)
results = np.random.choice(payouts, size=100_000, p=probs)
E_simulated = results.mean()
print(f"b) Simulated E[X]  = {E_simulated:.4f}")
print(f"   Difference:        {abs(E_analytical - E_simulated):.6f}")

# c) How many pulls to stabilise within ±0.10?
running_mean = np.cumsum(results) / np.arange(1, len(results)+1)
stable_at = np.argmax(np.abs(running_mean - E_analytical) < 0.10)
print(f"c) Stabilises within ±0.10 at pull: {stable_at}")

# YOUR TURN: Plot the running mean convergence
# fig, ax = plt.subplots(figsize=(10,4))
# ax.plot(running_mean[:5000], ...)
# ax.axhline(E_analytical, ...)


### Solution 2.2: Exploration Probability vs Sigma

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

sigmas = np.linspace(0.1, 3.0, 100)
mu = 3.0
threshold = 4.5

# P(a > 4.5) = 1 - CDF(4.5) for N(mu, sigma)
prob_explore = [1 - stats.norm.cdf(threshold, loc=mu, scale=s) for s in sigmas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sigmas, prob_explore, color='steelblue', linewidth=2)
ax.axvline(0.5, color='red', linestyle='--', label='σ=0.5 (low explore)')
ax.axvline(1.5, color='green', linestyle='--', label='σ=1.5 (moderate)')
ax.set_xlabel('Policy standard deviation σ')
ax.set_ylabel('P(action > 4.5)')
ax.set_title('How σ Controls Exploration Probability')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"At σ=0.5: P(explore) = {1-stats.norm.cdf(4.5, 3.0, 0.5):.4f}")
print(f"At σ=1.5: P(explore) = {1-stats.norm.cdf(4.5, 3.0, 1.5):.4f}")
print("RL implication: σ is the entropy of the policy — higher σ = more exploration")

# YOUR TURN: Find σ such that P(a > 4.5) = exactly 0.10
# Hint: use stats.norm.ppf or binary search over sigmas


### Solution 2.3: Markov Property and Stationary Distribution

In [ ]:
import numpy as np

P = np.array([
    [0.7, 0.2, 0.1],  # From Rainy
    [0.3, 0.4, 0.3],  # From Cloudy
    [0.2, 0.3, 0.5],  # From Sunny
])
states = ['Rainy', 'Cloudy', 'Sunny']

# a) 2-step transition matrix
P2 = P @ P   # matrix multiply
print("a) 2-step transition matrix P^2:")
print(np.round(P2, 4))
print(f"
P(Sunny in 2 days | Rainy today) = {P2[0,2]:.4f}")

# b) Stationary distribution via P^100
P_100 = np.linalg.matrix_power(P, 100)
stationary = P_100[0]   # all rows converge to same vector
print(f"
b) Stationary distribution: {dict(zip(states, stationary.round(4)))}")
print(f"   Rainy: {stationary[0]:.3f}, Cloudy: {stationary[1]:.3f}, Sunny: {stationary[2]:.3f}")

# Verify: stationary @ P should equal stationary
residual = np.max(np.abs(stationary @ P - stationary))
print(f"   Verification residual (π@P - π): {residual:.2e}  ← should be ~0")

# YOUR TURN: Compute the expected number of steps to reach 'Sunny' from 'Rainy'
# Hint: solve the system of linear equations for mean first passage times
